In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/04.gold-helpers"

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
        .withColumn("session_type", F.lit("RACE"))
        .filter((F.col("batch_id") == v_batch_id))
        .drop("ingestion_timestamp","source_file","race_name","race_date","batch_id","created_timestamp", "updated_timestamp")
        
)

sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
        .withColumn("session_type", F.lit("SPRINT"))
        .filter((F.col("batch_id") == v_batch_id))
        .drop("ingestion_timestamp","source_file","race_name","race_date","batch_id","created_timestamp", "updated_timestamp")
        
)

In [0]:
result_sprint_df = results_df.unionByName(sprints_df)
display(result_sprint_df.filter("season = 2025"))

In [0]:
fact_session_results_df = (
    result_sprint_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1,3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finish_position",
        "finish_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]

In [0]:
merge_consition = """
        t.season = s.season
        AND t.round = s.round
        AND t.constructor_id = s.constructor_id
        AND t.driver_id = s.driver_id
        AND t.session_type = s.session_type
"""

In [0]:
write_to_gold(
    df=fact_session_results_df,
    target_table=target_table,
    columns_to_update=columns_to_update,
    merge_condition=merge_consition
)

In [0]:
display(spark.table(target_table).limit(5))